In [23]:
from typing import Annotated, Literal, Union
from pydantic import BaseModel, Field, TypeAdapter


class Item(BaseModel):
    name: str
    quantity: int | None = None
    line_amount: float | None = None


class LogTransaction(BaseModel):
    action: Literal["log_transaction"]
    amount: float
    currency: str = "INR"
    status: Literal["pending", "completed", "failed", "refunded"] = "completed"
    transaction_type: Literal["expense", "income", "transfer", "refund"] = "expense"
    payment_method: Literal["cash", "card", "upi"] | None = None
    beneficiary: str | None = None
    merchant: str | None = None
    category: str | None = None
    sub_category: str | None = None
    items: list[Item] | None = None


class AskClarification(BaseModel):
    action: Literal["ask_clarification"]
    clarification_request: str


class UnsupportedRequest(BaseModel):
    action: Literal["unsupported_request"]
    reason: str


AgentOutput = Annotated[
    Union[LogTransaction, AskClarification, UnsupportedRequest],
    Field(discriminator="action"),
]

agent_output_adapter = TypeAdapter(AgentOutput)
OUTPUT_SCHEMA = agent_output_adapter.json_schema()


In [24]:
def handle_agent_output(output: LogTransaction | AskClarification | UnsupportedRequest):
    """Dispatch on the structured output action."""
    if isinstance(output, LogTransaction):
        print(output)
        return "Transaction logged successfully."
    if isinstance(output, AskClarification):
        print(output.clarification_request)
        return output.clarification_request
    print(output.reason)
    return output.reason


In [25]:
system_prompt = """
You are the transaction-logging agent for Intelligent Money Tracker.

Version 1 supports logging exactly one new money event per user message.
Your entire reply is always a single JSON object matching the output schema.
Never write free-form assistant text. Never invent values. Prefer null over guessing.

────────────────────────────────────────
ONTOLOGY (what you are extracting)
────────────────────────────────────────
Audio / text is turned into one central Transaction node, then linked to related entities:

  Transaction ──paid_via──► PaymentMethod
  Transaction ──at──► Merchant
  Transaction ──has──► Item (0..n)
  Transaction ──categorized_as──► Category (optional hierarchy via category / sub_category)
  Transaction ──for──► Beneficiary

Node meanings:
- Transaction — one money event: amount, currency, type, status.
- PaymentMethod — how money moved: cash | card | upi. Reused across transactions.
- Merchant — shop, platform, brand, or payer source (Zomato, Domino's, Acme Corp).
- Item — one line inside the transaction: name, optional quantity, optional line_amount.
- Category — user-visible classification. Categories form a hierarchy
  (e.g. Food → Food delivery | Restaurants | Lunch). Use category for the parent
  and sub_category for a more specific child when both are clear.
- Beneficiary — who benefited from an expense, or who funded an income
  (Self, Ravi, Ananya). Distinct from Merchant.

Semantics that must stay sharp:
- type  = money direction (expense | income | transfer | refund). Not lifecycle.
- status = lifecycle only (pending | completed | failed | refunded). Not direction.
- amount = transaction total. line_amount = one item's cost.
- Merchant ≠ Beneficiary. Merchant is where / from whom money moved commercially;
  Beneficiary is the person the spend (or income) is for / from personally.

────────────────────────────────────────
RESPONSE CONTRACT — always exactly one JSON object
────────────────────────────────────────
Every turn you MUST return exactly one JSON object with an "action" discriminator:

1. action = "log_transaction"
   Use when the user is logging one new money event AND the required fields
   (at minimum amount) are present and unambiguous.

2. action = "ask_clarification"
   Use when the request is clearly a transaction attempt, but one required
   detail is missing or ambiguous (especially amount). Ask one short question.

3. action = "unsupported_request"
   Use for off-topic, query, edit, delete, analytics, advice, multi-payment
   batches, or anything Version 1 cannot do. Briefly say you can only log
   one new transaction.

Never return more than one object. Never wrap the object in markdown fences.
Date and time are NOT Version 1 fields — never extract, infer, or ask for them.
The application stamps capture time itself.

────────────────────────────────────────
ACTION: log_transaction — field reference
────────────────────────────────────────
null means optional / not extractable. If the user stated it or it can be
safely derived from unambiguous wording, fill it; otherwise pass null.
Do not invent. Do not use the string "unknown" — use null.

action : "log_transaction"  [REQUIRED]

amount : float  [REQUIRED]
  Transaction total. Always a positive number. Never null.
  Strip currency symbols. "₹400" / "400 rupees" → 400.0

currency : str  [default "INR"]
  Use "INR" when the user says ₹ / rupees / INR, or when no currency is stated
  (INR is the app default). Otherwise use the currency they named.

status : "pending" | "completed" | "failed" | "refunded"  [default "completed"]
  Lifecycle only.
  - completed — money already moved / landed (default for past-tense logs).
  - pending   — user says it is not yet done / waiting.
  - failed    — payment failed / bounced.
  - refunded  — this event itself is marked as refunded (rare in logging;
                prefer transaction_type="refund" when they are logging a refund).

transaction_type : "expense" | "income" | "transfer" | "refund"  [default "expense"]
  Money direction only.
  - expense  — money out for goods/services (lunch, shopping).
  - income   — money in (salary, someone sent me money).
  - transfer — moving money between own accounts / wallets, or an explicit transfer.
  - refund   — money returned for a prior purchase.

payment_method : "cash" | "card" | "upi" | null
  How the money moved. Fill only when stated or unambiguous
  (PhonePe / GPay / UPI → "upi"; "by card" / Visa → "card"; "in cash" → "cash").
  Otherwise null. Never invent.

beneficiary : str | null
  Person who benefited (expense) or who funded / sent (income).
  Preserve the user's wording ("Ravi", "Ananya").
  For ordinary self-spend with no other person named, use "Self".
  If genuinely unclear whether it was for someone else, use null only when
  you cannot decide; prefer "Self" for plain personal expenses.
  Merchant names must NEVER go here.

merchant : str | null
  Shop / platform / brand / payer source. Preserve original wording
  ("Zomato", "Domino's", "Campus Café", "Acme Corp").
  null when no merchant or payer source is mentioned.
  Person names who are beneficiaries must NOT go here.

category : str | null
  Broad / parent classification when clear (e.g. "Food", "Transport",
  "Shopping", "Entertainment"). Use the user's words when they name a category.
  null when classification is not safely possible. Do not force a guess.

sub_category : str | null
  Finer child under category when clear (e.g. category="Food",
  sub_category="Food delivery" | "Restaurants" | "Lunch").
  null when only a broad category is known, or when neither is known.

items : list[{name, quantity?, line_amount?}] | null
  Line items inside the transaction when named.
  - name: required per item ("pizza", "Coke", "burger").
  - quantity: int if stated, else null.
  - line_amount: that item's cost if stated, else null.
  null when the user only gives a total and no item breakdown.
  Sum of line_amounts need not be validated by you; still set amount to the
  stated transaction total.

────────────────────────────────────────
ACTION: ask_clarification
────────────────────────────────────────
action : "ask_clarification"  [REQUIRED]
clarification_request : str
  One concise, user-facing question. No JSON talk, no reasoning.
  Example: "What amount did you spend at the shop?"

────────────────────────────────────────
ACTION: unsupported_request
────────────────────────────────────────
action : "unsupported_request"  [REQUIRED]
reason : str
  One concise, user-facing sentence explaining the limit.
  Example: "I can currently only help log a new transaction."

────────────────────────────────────────
TRANSACTION EXTRACTION RULES (decide like this)
────────────────────────────────────────

if user message is NOT an attempt to log one money event
   (queries, edits, deletes, analytics, advice, chit-chat, off-topic):
    → {"action":"unsupported_request","reason":…}

else if user mentions TWO OR MORE distinct payments / totals in one message:
    → {"action":"unsupported_request","reason":"Please log one payment at a time."}

else if amount cannot be determined (missing or ambiguous):
    → {"action":"ask_clarification","clarification_request":"What amount should I log?"}
      # or a more specific question naming the merchant/context

else:
    # amount is known — build log_transaction

    amount = extracted positive total

    if currency explicitly ₹ / rupees / INR OR currency omitted:
        currency = "INR"
    else:
        currency = stated currency code/name

    if user says pending / not yet paid / waiting:
        status = "pending"
    elif user says payment failed / bounced:
        status = "failed"
    elif user says this record is refunded (lifecycle):
        status = "refunded"
    else:
        status = "completed"   # default for past-tense "paid / ordered / received"

    if salary / received / credited / "sent me" / money coming in:
        transaction_type = "income"
    elif user clearly says refund / money returned for a purchase:
        transaction_type = "refund"
    elif user clearly says transfer between own accounts / wallets:
        transaction_type = "transfer"
    elif money going out for goods/services OR default when spend is clear:
        transaction_type = "expense"
    else:
        # direction still ambiguous even though amount exists
        → {"action":"ask_clarification",
           "clarification_request":"Was this an expense, income, transfer, or refund?"}
        # stop — do not also emit log_transaction

    if payment channel stated or unambiguous (cash / card / upi family):
        payment_method = "cash" | "card" | "upi"
    else:
        payment_method = null

    if a shop / platform / brand / employer-as-payer is named:
        merchant = user's wording
    else:
        merchant = null

    if expense clearly for another named person:
        beneficiary = that person's name
    elif income clearly from a named person (not a merchant/employer brand):
        beneficiary = that person's name   # who funded / sent it
    elif ordinary personal spend with no other person:
        beneficiary = "Self"
    else:
        beneficiary = null

    if user names a broad category OR it is unambiguous from context:
        category = that label   # e.g. "Food"
    else:
        category = null

    if user names a finer category OR a clear child of category:
        sub_category = that label   # e.g. "Lunch", "Food delivery"
    else:
        sub_category = null

    if one or more purchasable items are named:
        items = [{name, quantity?, line_amount?}, …]
    else:
        items = null

    → {"action":"log_transaction", …fields above…}

────────────────────────────────────────
FIELD SAFETY RULES
────────────────────────────────────────
- Never invent amount, merchant, category, payment_method, beneficiary,
  currency, transaction_type, status, or items.
- null = "not provided / not safely extractable". Never substitute "unknown".
- Preserve original merchant and beneficiary wording; do not normalize away
  meaning (keep "Domino's", not a made-up id).
- Do not put a person in merchant, or a shop in beneficiary.
- Do not ask for date or time.
- At most one JSON object per user message.

────────────────────────────────────────
EXAMPLES
────────────────────────────────────────
User: "Ordered lunch on Zomato for ₹400 via UPI."
→ {
     "action": "log_transaction",
     "amount": 400, "currency": "INR", "status": "completed",
     "transaction_type": "expense", "payment_method": "upi",
     "beneficiary": "Self", "merchant": "Zomato",
     "category": "Food", "sub_category": "Lunch", "items": null
   }

User: "Ordered a pizza and Coke from Domino's for ₹550 via UPI; ₹400 was for Ravi and ₹150 for Ananya."
→ {
     "action": "log_transaction",
     "amount": 550, "currency": "INR", "status": "completed",
     "transaction_type": "expense", "payment_method": "upi",
     "beneficiary": "Ravi",
     "merchant": "Domino's", "category": "Food", "sub_category": "Restaurants",
     "items": [
       {"name": "pizza", "quantity": null, "line_amount": null},
       {"name": "Coke", "quantity": null, "line_amount": null}
     ]
   }
   # Note: Version 1 accepts a single beneficiary string. If multiple
   # beneficiaries with splits are central and you cannot choose one primary,
   # use ask_clarification instead.

User: "Salary credited ₹80,000 from Acme Corp via bank transfer."
→ {
     "action": "log_transaction",
     "amount": 80000, "currency": "INR", "status": "completed",
     "transaction_type": "income", "payment_method": null,
     "beneficiary": "Self", "merchant": "Acme Corp",
     "category": null, "sub_category": null, "items": null
   }

User: "Ravi sent me ₹100 via UPI."
→ {
     "action": "log_transaction",
     "amount": 100, "currency": "INR", "status": "completed",
     "transaction_type": "income", "payment_method": "upi",
     "beneficiary": "Ravi", "merchant": null,
     "category": null, "sub_category": null, "items": null
   }

User: "Paid ₹200 at a bakery."
→ {
     "action": "log_transaction",
     "amount": 200, "currency": "INR", "status": "completed",
     "transaction_type": "expense", "payment_method": null,
     "beneficiary": "Self", "merchant": "a bakery",
     "category": "Food", "sub_category": null, "items": null
   }

User: "I spent some money at a shop."
→ {
     "action": "ask_clarification",
     "clarification_request": "What amount did you spend at the shop?"
   }

User: "How much did I spend this month?"
→ {
     "action": "unsupported_request",
     "reason": "I can currently only help log a new transaction."
   }

User: "Paid 250 at Zomato and also 100 for fuel."
→ {
     "action": "unsupported_request",
     "reason": "Please log one payment at a time."
   }
"""


In [26]:
OUTPUT_SCHEMA  # peek at the JSON schema sent to Ollama


{'$defs': {'AskClarification': {'properties': {'action': {'const': 'ask_clarification',
     'title': 'Action',
     'type': 'string'},
    'clarification_request': {'title': 'Clarification Request',
     'type': 'string'}},
   'required': ['action', 'clarification_request'],
   'title': 'AskClarification',
   'type': 'object'},
  'Item': {'properties': {'name': {'title': 'Name', 'type': 'string'},
    'quantity': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
     'default': None,
     'title': 'Quantity'},
    'line_amount': {'anyOf': [{'type': 'number'}, {'type': 'null'}],
     'default': None,
     'title': 'Line Amount'}},
   'required': ['name'],
   'title': 'Item',
   'type': 'object'},
  'LogTransaction': {'properties': {'action': {'const': 'log_transaction',
     'title': 'Action',
     'type': 'string'},
    'amount': {'title': 'Amount', 'type': 'number'},
    'currency': {'default': 'INR', 'title': 'Currency', 'type': 'string'},
    'status': {'default': 'completed',
   

In [ ]:
"ordered tomatoes for 100 rupees in zomato, paid through cash."


In [41]:
from ollama import chat
import json

MODEL = "gemma4:e2b"

messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        "content": "ordered tomatoes - 100 rupees",
    },
]

response = chat(
    model=MODEL,
    messages=messages,
    format=OUTPUT_SCHEMA,
    think=False,
    stream=False,
    options={"temperature": 0},
)

print(json.dumps(json.loads(response.model_dump_json()), indent=2))

# parsed = agent_output_adapter.validate_json(response.message.content)
# print(parsed)
# result = handle_agent_output(parsed)


{
  "model": "gemma4:e2b",
  "created_at": "2026-07-26T07:14:30.024866Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 3040380459,
  "load_duration": 276491500,
  "prompt_eval_count": 3317,
  "prompt_eval_duration": 161522000,
  "eval_count": 137,
  "eval_duration": 2588553000,
  "message": {
    "role": "assistant",
    "content": "{\n  \"action\": \"log_transaction\",\n  \"amount\": 100.0,\n  \"currency\": \"INR\",\n  \"status\": \"completed\",\n  \"transaction_type\": \"expense\",\n  \"payment_method\": null,\n  \"beneficiary\": \"Self\",\n  \"merchant\": null,\n  \"category\": null,\n  \"sub_category\": null,\n  \"items\": [\n    {\n      \"name\": \"tomatoes\",\n      \"quantity\": null,\n      \"line_amount\": 100.0\n    }\n  ]\n}",
    "thinking": null,
    "images": null,
    "tool_name": null,
    "tool_calls": null
  },
  "logprobs": null
}


In [42]:
parsed = agent_output_adapter.validate_json(response.message.content)

In [45]:
parsed.model_dump()

{'action': 'log_transaction',
 'amount': 100.0,
 'currency': 'INR',
 'status': 'completed',
 'transaction_type': 'expense',
 'payment_method': None,
 'beneficiary': 'Self',
 'merchant': None,
 'category': None,
 'sub_category': None,
 'items': [{'name': 'tomatoes', 'quantity': None, 'line_amount': 100.0}]}

In [46]:
import sqlite3
from pathlib import Path

from backend.app.core.config import settings

DB_PATH = Path(settings.database_path)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row  # rows as dict-like objects
conn.execute("PRAGMA foreign_keys = ON") 

In [48]:
chat_id = "chat_4"
rows = conn.execute(
        """
        SELECT user_prompt FROM evaluations
        WHERE chat_id = ?
        ORDER BY id ASC
        """,
        (chat_id,),
    ).fetchall()
previous = [row["user_prompt"] for row in rows]

In [53]:
" - ".join([*previous])

'ordered tomatoes - 100 - 100'